# Pipeline Data Cleaning & Data Transformation NASA Meteorite & Asteroid Data

Pipeline ini mengimplementasikan pengolahan data end-to-end dengan pemisahan tahapan yang tegas:
1. **Tahap 1: Data Cleaning (Pembersihan Data Mentah)**:
   - Audit trail pembersihan data (*missing values*, anomali koordinat *Null Island* `(0.0, 0.0)`, tahun masa depan `> 2026`, massa `<= 0 g`).
   - Parsing koordinat kardinal CNEOS Fireballs (format teks `NSEW` menjadi derajat desimal numerik bertanda).
   - Filtering anomali fisik dan validasi integritas data.
2. **Tahap 2: Data Transformation (Flattening, Standardisasi Skema, Feature Engineering & Export Seragam)**:
   - **Flattening**: Meratakan struktur hierarkis/nested JSON (NASA NeoWs, JPL Sentry, dan SBDB CAD) menjadi tabel 2 dimensi datar.
   - **Standardisasi Skema**: 
     - **Nama Kolom**: Penyeragaman ke konvensi baku `snake_case` lintas dataset (`latitude`, `longitude`, `designation`, `mass_g`, `mass_kg`, `altitude_km`, `velocity_kms`, dll).
     - **Tipe Data**: Penetapan tipe data eksplisit (`float64`, `int64`, `str`, `bool`) untuk konsistensi analitik.
     - **Format Tanggal**: Penyeragaman seluruh komponen waktu ke format baku ISO UTC `YYYY-MM-DD HH:MM:SS` pada kolom `datetime_utc`.
     - **Satuan**: Standarisasi besaran fisik (derajat desimal, km, km/s, Lunar Distances, Joule, Ton TNT, g/cm³, MPa).
     - **Encoding**: Seluruh file disimpan dengan standar encoding `UTF-8` eksplisit.
   - **Feature Engineering & Material Physics**: Klasifikasi 4 kelas material fisik batuan luar angkasa, densitas, kuat luluh (*yield strength*), rezim dampak (*airburst* vs *crater*), log massa, dekade, dan estimasi energi kinetik hantaman (Ton TNT) untuk meteorit Indonesia.
   - **Penyeragaman Format File Keluaran**: Seluruh dataset yang telah ditransformasikan diekspor ke satu format seragam, yaitu **CSV (`.csv`)** di folder `datasets/transformed/`.

In [1]:
import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

# Inisialisasi path direktori proyek
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
TRANSFORMED_DIR = DATASETS_DIR / "transformed"
TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Datasets dir : {DATASETS_DIR}")
print(f"Output dir   : {TRANSFORMED_DIR}")

Project root : /home/saaandd/web/hacknasa
Datasets dir : /home/saaandd/web/hacknasa/datasets
Output dir   : /home/saaandd/web/hacknasa/datasets/transformed


## Ingest Data Mentah (Raw Ingestion)
Memuat dataset mentah dari penyimpanan lokal:
- `Meteorite_Landings.csv`: 45.716 baris rekaman meteorit global.
- `cneos_fireball_data.csv`: Rekaman fireball/bolide atmosferik dari sensor CNEOS NASA.
- `datasets/cneos_sentry/sentry.json`: Snapshot data pemantauan risiko tabrakan asteroid JPL Sentry.
- `datasets/sbdb_cad/cad.json`: Snapshot rekaman lintasan asteroid dekat Bumi (*close-approaches*).
- `datasets/nasa_neows/feed.json`: Snapshot nested JSON Near-Earth Objects (NeoWs) NASA.

In [2]:
# 1. Load Raw Meteorite Landings
raw_meteorites_path = DATASETS_DIR / "Meteorite_Landings.csv"
df_raw_meteorites = pd.read_csv(raw_meteorites_path)
print(f"[Load] Meteorite Landings Raw : {len(df_raw_meteorites):,} rows")

# 2. Load CNEOS Fireball Data
raw_fireball_path = DATASETS_DIR / "cneos_fireball_data.csv"
df_raw_fireball = pd.read_csv(raw_fireball_path, header=[0, 1])
print(f"[Load] CNEOS Fireballs Raw    : {len(df_raw_fireball):,} rows")

# 3. Load API Snapshots (Nested JSON)
with open(DATASETS_DIR / "cneos_sentry" / "sentry.json") as f:
    sentry_raw = json.load(f)
with open(DATASETS_DIR / "sbdb_cad" / "cad.json") as f:
    cad_raw = json.load(f)
with open(DATASETS_DIR / "nasa_neows" / "feed.json") as f:
    neows_raw = json.load(f)

print(f"[Load] Sentry Tracked Objects : {len(sentry_raw.get('data', [])):,} objects")
print(f"[Load] CAD Close Approaches   : {len(cad_raw.get('data', [])):,} events")
print(f"[Load] NeoWs Feed NEO Days    : {len(neows_raw.get('near_earth_objects', {})):,} days")

[Load] Meteorite Landings Raw : 45,716 rows
[Load] CNEOS Fireballs Raw    : 1,074 rows
[Load] Sentry Tracked Objects : 2,251 objects
[Load] CAD Close Approaches   : 1,097 events
[Load] NeoWs Feed NEO Days    : 7 days


---
# TAHAP 1: DATA CLEANING (PEMBERSIHAN DATA MENTAH)

Pada tahap ini dilakukan proses pembersihan murni:
1. **Pembersihan Missing Values**: Menghapus baris tanpa nilai koordinat dan massa.
2. **Pembersihan Anomali Koordinat *Null Island***: Menghapus titik jatuh palsu pada koordinat `(0.0, 0.0)` di Teluk Guinea.
3. **Pembersihan Anomali Nilai**: Menghapus data tahun masa depan (`> 2026`) dan massa tidak logis (`<= 0 g`).
4. **Pembersihan & Parsing Koordinat Kardinal**: Mengubah format koordinat teks kardinal (`NSEW`) pada dataset Fireballs menjadi desimal numerik bertanda.

In [3]:
# ==============================================================================
# 1.1 Data Cleaning: Meteorite Landings Funnel
# ==============================================================================
raw_count = len(df_raw_meteorites)

# 1. Drop Missing Values
df_step1 = df_raw_meteorites.dropna(subset=["reclat", "reclong", "mass (g)"]).copy()
dropped_nan = raw_count - len(df_step1)

# 2. Drop Null Island (0.0, 0.0)
df_step2 = df_step1[~((df_step1["reclat"] == 0.0) & (df_step1["reclong"] == 0.0))].copy()
dropped_null_island = len(df_step1) - len(df_step2)

# 3. Drop Anomali Tahun (> 2026) dan Massa (<= 0 g)
df_clean_meteorites = df_step2[
    (df_step2["year"] <= 2026) & (df_step2["mass (g)"] > 0)
].copy()
dropped_anomalies = len(df_step2) - len(df_clean_meteorites)

final_count = len(df_clean_meteorites)
retention_rate = (final_count / raw_count) * 100

print("=== DATA CLEANING AUDIT TRAIL: METEORITE LANDINGS ===")
print(f"1. Raw Initial Records  : {raw_count:,} rows")
print(f"2. Dropped Missing NaN  : -{dropped_nan:,} rows -> Remaining {len(df_step1):,}")
print(f"3. Dropped Null Island  : -{dropped_null_island:,} rows -> Remaining {len(df_step2):,}")
print(f"4. Dropped Anomalies    : -{dropped_anomalies:,} rows -> Remaining {final_count:,}")
print(f"-> TOTAL CLEAN DATASET  : {final_count:,} rows ({retention_rate:.2f}% retention)")

assert final_count == 31911, f"Expected 31911 rows, got {final_count}"

=== DATA CLEANING AUDIT TRAIL: METEORITE LANDINGS ===
1. Raw Initial Records  : 45,716 rows
2. Dropped Missing NaN  : -7,434 rows -> Remaining 38,282
3. Dropped Null Island  : -6,214 rows -> Remaining 32,068
4. Dropped Anomalies    : -157 rows -> Remaining 31,911
-> TOTAL CLEAN DATASET  : 31,911 rows (69.80% retention)


### 1.2 Data Cleaning: CNEOS Fireballs (Parsing Koordinat Kardinal)
Mengonversi format koordinat teks kardinal seperti `41.8S` menjadi `-41.8` dan `173.1W` menjadi `-173.1`. Hanya rekaman dengan koordinat yang valid yang dipertahankan.

In [4]:
# ==============================================================================
# 1.2 Data Cleaning: CNEOS Fireballs
# ==============================================================================
def parse_coord(val):
    if pd.isna(val) or not str(val).strip():
        return np.nan
    val_str = str(val).strip()
    m = re.match(r"^([0-9.]+)([NSEW])$", val_str)
    if not m:
        return np.nan
    num = float(m.group(1))
    direction = m.group(2)
    if direction in ["S", "W"]:
        num = -num
    return num

fireball_records = []
for _, row in df_raw_fireball.iterrows():
    dt = row.iloc[0]
    lat = parse_coord(row.iloc[1])
    lon = parse_coord(row.iloc[2])
    alt = float(row.iloc[3]) if pd.notna(row.iloc[3]) and str(row.iloc[3]).strip() else None
    energy_j = float(row.iloc[7]) if pd.notna(row.iloc[7]) and str(row.iloc[7]).strip() else None
    energy_kt = float(row.iloc[8]) if pd.notna(row.iloc[8]) and str(row.iloc[8]).strip() else None
    
    if pd.notna(lat) and pd.notna(lon):
        fireball_records.append({
            "raw_datetime": str(dt).strip(),
            "latitude": float(lat),
            "longitude": float(lon),
            "altitude_km": float(alt) if alt is not None else np.nan,
            "energy_joules": float(energy_j) if energy_j is not None else np.nan,
            "energy_kilotons_tnt": float(energy_kt) if energy_kt is not None else np.nan
        })

df_clean_fireball = pd.DataFrame(fireball_records)
print(f"CNEOS Fireballs Cleaned (Valid Coords): {len(df_clean_fireball)} events (dari {len(df_raw_fireball)} raw)")

CNEOS Fireballs Cleaned (Valid Coords): 888 events (dari 1074 raw)


---
# TAHAP 2: DATA TRANSFORMATION

Tahap transformasi ini mencakup dua pilar utama:
1. **Flattening**: Meratakan struktur JSON bertingkat (*nested structures*) dari NASA NeoWs, JPL Sentry, dan SBDB CAD menjadi tabel 2 dimensi flat.
2. **Standardisasi Skema**: 
   - **Nama Kolom**: Menyeragamkan penamaan seluruh kolom menggunakan standar `snake_case` semantik (`latitude`, `longitude`, `designation`, `mass_g`, `mass_kg`, `altitude_km`, dll).
   - **Tipe Data**: Memastikan tipe data eksplisit (`float64`, `int64`, `str`, `bool`).
   - **Format Tanggal**: Menyeragamkan format waktu di seluruh dataset menjadi standar ISO UTC `YYYY-MM-DD HH:MM:SS` pada kolom `datetime_utc`.
   - **Satuan**: Standarisasi satuan besaran fisik (derajat desimal, km, km/s, Lunar Distances, Joule, Ton TNT, g/cm³, MPa).
   - **Encoding**: Seluruh file disimpan dengan standar encoding `UTF-8` eksplisit.

### 2.1 Flattening: Meratakan Struktur JSON Nested ke Tabel Flat
Meratakan data hierarkis dari:
1. **NASA NeoWs API**: Membongkar objek bertingkat `estimated_diameter` (kilometers) dan array `close_approach_data` (rel speed & miss distance).
2. **JPL Sentry API**: Meratakan array `data` objek risiko dampak.
3. **SBDB CAD API**: Menggabungkan array `fields` dengan baris data `data` menjadi tabel flat.

In [5]:
# ==============================================================================
# 2.1 Flattening: NASA NeoWs (Deeply Nested JSON -> Flat Table)
# ==============================================================================
neows_flat_records = []
for date_str, neo_list in neows_raw.get("near_earth_objects", {}).items():
    for neo in neo_list:
        est_diam = neo.get("estimated_diameter", {}).get("kilometers", {})
        ca_list = neo.get("close_approach_data", [])
        ca = ca_list[0] if len(ca_list) > 0 else {}
        rel_vel = ca.get("relative_velocity", {})
        miss_dist = ca.get("miss_distance", {})
        
        # Format raw approach date
        raw_ca_dt = ca.get("close_approach_date_full") or (
            ca.get("close_approach_date") + " 00:00:00" if ca.get("close_approach_date") else None
        )
        
        neows_flat_records.append({
            "neo_id": str(neo.get("id")),
            "name": str(neo.get("name")),
            "designation": str(neo.get("designation", neo.get("name"))),
            "absolute_magnitude_h": float(neo.get("absolute_magnitude_h")) if neo.get("absolute_magnitude_h") is not None else np.nan,
            "is_potentially_hazardous": bool(neo.get("is_potentially_hazardous_asteroid", False)),
            "is_sentry_object": bool(neo.get("is_sentry_object", False)),
            "diameter_min_km": float(est_diam.get("estimated_diameter_min")) if est_diam.get("estimated_diameter_min") is not None else np.nan,
            "diameter_max_km": float(est_diam.get("estimated_diameter_max")) if est_diam.get("estimated_diameter_max") is not None else np.nan,
            "raw_datetime": raw_ca_dt,
            "velocity_kms": float(rel_vel.get("kilometers_per_second")) if rel_vel.get("kilometers_per_second") is not None else np.nan,
            "distance_km": float(miss_dist.get("kilometers")) if miss_dist.get("kilometers") is not None else np.nan,
            "distance_ld": float(miss_dist.get("lunar")) if miss_dist.get("lunar") is not None else np.nan,
            "orbiting_body": str(ca.get("orbiting_body", "Earth"))
        })

df_neows = pd.DataFrame(neows_flat_records)
print(f"Flattened NeoWs Asteroids : {len(df_neows)} rekaman")

# ==============================================================================
# 2.1 Flattening: JPL Sentry
# ==============================================================================
sentry_flat_records = []
for obj in sentry_raw.get("data", []):
    sentry_flat_records.append({
        "designation": str(obj.get("des")),
        "fullname": str(obj.get("fullname")),
        "impact_probability": float(obj.get("ip", 0.0)),
        "diameter_km": float(obj.get("diameter")) if obj.get("diameter") else np.nan,
        "palermo_scale_max": float(obj.get("ps_max")) if obj.get("ps_max") else np.nan,
        "torino_scale_max": int(obj.get("ts_max")) if obj.get("ts_max") else 0,
        "velocity_inf_kms": float(obj.get("v_inf")) if obj.get("v_inf") else np.nan,
        "year_range": str(obj.get("range")),
        "raw_last_obs": obj.get("last_obs")
    })
df_clean_sentry = pd.DataFrame(sentry_flat_records).sort_values(by="impact_probability", ascending=False)
print(f"Flattened Sentry Records  : {len(df_clean_sentry)} objek")

# ==============================================================================
# 2.1 Flattening: SBDB CAD
# ==============================================================================
cad_fields = cad_raw.get("fields", [])
cad_rows = cad_raw.get("data", [])
df_clean_cad = pd.DataFrame(cad_rows, columns=cad_fields)
print(f"Flattened CAD Records     : {len(df_clean_cad)} rekaman")

Flattened NeoWs Asteroids : 44 rekaman
Flattened Sentry Records  : 2251 objek
Flattened CAD Records     : 1097 rekaman


### 2.2 Standardisasi Skema: Format Tanggal & Waktu (ISO UTC: `YYYY-MM-DD HH:MM:SS`)
Mengecek dan menyeragamkan kolom waktu pada setiap dataset ke format baku `datetime_utc`.

In [6]:
# ==============================================================================
# 2.2 Penyeragaman Format Waktu ke "YYYY-MM-DD HH:MM:SS" (UTC)
# ==============================================================================
# 1. Meteorite Landings: tahun integer -> "YYYY-01-01 00:00:00"
df_clean_meteorites["datetime_utc"] = df_clean_meteorites["year"].apply(
    lambda y: f"{int(y):04d}-01-01 00:00:00" if pd.notna(y) else None
)

# 2. CNEOS Fireballs: 'raw_datetime' -> "YYYY-MM-DD HH:MM:SS"
df_clean_fireball["datetime_utc"] = pd.to_datetime(
    df_clean_fireball["raw_datetime"], errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")
df_clean_fireball.drop(columns=["raw_datetime"], inplace=True)

# 3. JPL Sentry: 'raw_last_obs' (YYYY-MM-DD) -> "YYYY-MM-DD 00:00:00"
df_clean_sentry["datetime_utc"] = pd.to_datetime(
    df_clean_sentry["raw_last_obs"], errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")
df_clean_sentry.drop(columns=["raw_last_obs"], inplace=True)

# 4. SBDB CAD: 'cd' (misal '2026-Oct-15 14:38') -> "YYYY-MM-DD HH:MM:SS"
df_clean_cad["datetime_utc"] = pd.to_datetime(
    df_clean_cad["cd"], format="%Y-%b-%d %H:%M", errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")

# 5. NASA NeoWs: 'raw_datetime' -> "YYYY-MM-DD HH:MM:SS"
df_neows["datetime_utc"] = pd.to_datetime(
    df_neows["raw_datetime"], errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")
df_neows.drop(columns=["raw_datetime"], inplace=True)

# Verifikasi keseragaman waktu
time_audit = pd.DataFrame([
    {"Dataset": "Meteorite Landings", "Format Asli": "YYYY (Numeric Year)", "Kolom Standar": "datetime_utc", "Sampel Format Baku": df_clean_meteorites["datetime_utc"].iloc[0]},
    {"Dataset": "CNEOS Fireballs", "Format Asli": "Peak Brightness Date/Time", "Kolom Standar": "datetime_utc", "Sampel Format Baku": df_clean_fireball["datetime_utc"].iloc[0]},
    {"Dataset": "JPL Sentry", "Format Asli": "last_obs (YYYY-MM-DD)", "Kolom Standar": "datetime_utc", "Sampel Format Baku": df_clean_sentry["datetime_utc"].iloc[0]},
    {"Dataset": "SBDB CAD", "Format Asli": "cd (YYYY-Mon-DD HH:MM)", "Kolom Standar": "datetime_utc", "Sampel Format Baku": df_clean_cad["datetime_utc"].iloc[0]},
    {"Dataset": "NASA NeoWs", "Format Asli": "close_approach_date_full", "Kolom Standar": "datetime_utc", "Sampel Format Baku": df_neows["datetime_utc"].iloc[0]},
])
print("=== AUDIT PENYERAGAMAN FORMAT WAKTU (TARGET: YYYY-MM-DD HH:MM:SS) ===")
print(time_audit.to_string(index=False))

=== AUDIT PENYERAGAMAN FORMAT WAKTU (TARGET: YYYY-MM-DD HH:MM:SS) ===
           Dataset               Format Asli Kolom Standar  Sampel Format Baku
Meteorite Landings       YYYY (Numeric Year)  datetime_utc 1880-01-01 00:00:00
   CNEOS Fireballs Peak Brightness Date/Time  datetime_utc 2026-10-04 03:14:45
        JPL Sentry     last_obs (YYYY-MM-DD)  datetime_utc 2022-08-24 00:00:00
          SBDB CAD    cd (YYYY-Mon-DD HH:MM)  datetime_utc 2026-10-15 14:38:00
        NASA NeoWs  close_approach_date_full  datetime_utc 2026-10-29 17:05:00


### 2.3 Standardisasi Skema: Penamaan Kolom (snake_case), Satuan, & Tipe Data
1. Menghilangkan spasi dan tanda kurung (misal: `mass (g)` diubah menjadi `mass_g`, dan ditambahkan `mass_kg`).
2. Menyeragamkan koordinat spasial ke `latitude` dan `longitude` secara konsisten di seluruh dataset spasial.
3. Menghilangkan kolom redundant atau unstandardized (misal `GeoLocation` format string kasar).
4. Menstandarisasi penamaan metrik jarak (`distance_au`, `distance_ld`, `distance_km`), kecepatan (`velocity_kms`), dan magnitudo (`absolute_magnitude_h`).

In [7]:
# ==============================================================================
# 2.3 Standardisasi Skema Kolom & Satuan
# ==============================================================================
# 1. Meteorite Landings
df_clean_meteorites = df_clean_meteorites.rename(columns={
    "reclat": "latitude",
    "reclong": "longitude",
    "mass (g)": "mass_g",
    "fall": "fall_type"
})
df_clean_meteorites["mass_kg"] = df_clean_meteorites["mass_g"] / 1000.0
if "GeoLocation" in df_clean_meteorites.columns:
    df_clean_meteorites.drop(columns=["GeoLocation"], inplace=True)

# 2. SBDB CAD: Rename, Unit Conversion & Clean Redundant Cols
df_clean_cad["distance_au"] = pd.to_numeric(df_clean_cad["dist"], errors="coerce")
df_clean_cad["distance_min_au"] = pd.to_numeric(df_clean_cad["dist_min"], errors="coerce")
df_clean_cad["distance_max_au"] = pd.to_numeric(df_clean_cad["dist_max"], errors="coerce")
df_clean_cad["distance_ld"] = df_clean_cad["distance_au"] * 389.1703
df_clean_cad["distance_km"] = df_clean_cad["distance_au"] * 149597870.7
df_clean_cad["velocity_rel_kms"] = pd.to_numeric(df_clean_cad["v_rel"], errors="coerce")
df_clean_cad["velocity_inf_kms"] = pd.to_numeric(df_clean_cad["v_inf"], errors="coerce")
df_clean_cad["absolute_magnitude_h"] = pd.to_numeric(df_clean_cad["h"], errors="coerce")

# Hapus kolom mentah yang sudah ditransformasi ke unit standar
df_clean_cad = df_clean_cad.rename(columns={"des": "designation"})
drop_cad_raw = [c for c in ["dist", "dist_min", "dist_max", "v_rel", "v_inf", "h", "cd"] if c in df_clean_cad.columns]
df_clean_cad.drop(columns=drop_cad_raw, inplace=True)

print("Skema kolom dan satuan berhasil distandarisasi.")

Skema kolom dan satuan berhasil distandarisasi.


### 2.4 Feature Engineering & Material Physics
Menghitung klasifikasi 4 kelas batuan luar angkasa, densitas, kuat luluh mekanik, rezim dampak, transformasi log massa, dekade, filter meteorit Indonesia beserta estimasi energi kinetik hantaman (Ton TNT).

In [8]:
# ==============================================================================
# 2.4 Feature Engineering: Material Physical Classes
# ==============================================================================
def map_composition(rc):
    rc_str = str(rc).strip()
    if any(k in rc_str for k in ["Iron", "Pallasite", "Mesosiderite"]):
        return "Iron / Stony-Iron"
    achondrites = ["Achondrite", "Aubrite", "Diogenite", "Eucrite", "Howardite", "Ureilite", "Angrite", "Brachinite", "Martian", "Lunar"]
    if any(a in rc_str for a in achondrites):
        return "Achondrite"
    if rc_str.startswith("C") and not rc_str.startswith("Chondrite-fusion") and not rc_str.startswith("Chondrite-ung"):
        return "Carbonaceous Chondrite"
    return "Ordinary Chondrite"

df_clean_meteorites["material_class"] = df_clean_meteorites["recclass"].apply(map_composition)

PHYSICAL_PARAMS = {
    "Iron / Stony-Iron": {"density_g_cm3": 7.8, "yield_strength_mpa": 350, "regime": "Surface Crater Impact"},
    "Achondrite": {"density_g_cm3": 3.2, "yield_strength_mpa": 65, "regime": "Mid-Atmosphere Breakup (15-25 km)"},
    "Carbonaceous Chondrite": {"density_g_cm3": 2.2, "yield_strength_mpa": 5, "regime": "High-Altitude Airburst (>35 km)"},
    "Ordinary Chondrite": {"density_g_cm3": 3.4, "yield_strength_mpa": 35, "regime": "Tropospheric Shockwave Airburst (15-30 km)"}
}

df_clean_meteorites["density_g_cm3"] = df_clean_meteorites["material_class"].map(lambda c: PHYSICAL_PARAMS[c]["density_g_cm3"]).astype(float)
df_clean_meteorites["yield_strength_mpa"] = df_clean_meteorites["material_class"].map(lambda c: PHYSICAL_PARAMS[c]["yield_strength_mpa"]).astype(float)
df_clean_meteorites["impact_regime"] = df_clean_meteorites["material_class"].map(lambda c: PHYSICAL_PARAMS[c]["regime"]).astype(str)

# Feature Transformations
df_clean_meteorites["log10_mass_g"] = np.log10(df_clean_meteorites["mass_g"]).astype(float)
df_clean_meteorites["abs_latitude"] = df_clean_meteorites["latitude"].abs().astype(float)
df_clean_meteorites["decade"] = (df_clean_meteorites["year"] // 10 * 10).astype(int)
df_clean_meteorites["is_fell"] = (df_clean_meteorites["fall_type"] == "Fell").astype(int)
df_clean_meteorites["is_massive_outlier"] = (df_clean_meteorites["log10_mass_g"] > 4.5448).astype(bool)

# Subsetting & Feature Engineering: Meteorit Indonesia
df_indonesia = df_clean_meteorites[
    (df_clean_meteorites["latitude"] >= -11.5) & (df_clean_meteorites["latitude"] <= 6.0) &
    (df_clean_meteorites["longitude"] >= 95.0) & (df_clean_meteorites["longitude"] <= 141.0)
].sort_values(by="mass_g", ascending=False).copy()

def calc_energy(mass_g, velocity_ms=17000):
    mass_kg = mass_g / 1000.0
    joules = 0.5 * mass_kg * (velocity_ms ** 2)
    tons_tnt = joules / 4.184e9
    return round(tons_tnt, 2)

df_indonesia["energy_tons_tnt"] = df_indonesia["mass_g"].apply(calc_energy).astype(float)

print(f"Distribusi Kelas Material Meteorit:")
print(df_clean_meteorites["material_class"].value_counts())
print(f"\nTotal Rekaman Meteorit Indonesia: {len(df_indonesia)} objek")
for idx, row in df_indonesia.head(3).iterrows():
    print(f"- {row['name']} ({row['year']:.0f}): {row['mass_kg']:.2f} kg, {row['material_class']}, {row['energy_tons_tnt']} Tons TNT, Waktu: {row['datetime_utc']}")

Distribusi Kelas Material Meteorit:
material_class
Ordinary Chondrite        28988
Iron / Stony-Iron          1113
Achondrite                  985
Carbonaceous Chondrite      825
Name: count, dtype: int64

Total Rekaman Meteorit Indonesia: 19 objek
- Prambanan (1797): 500.00 kg, Iron / Stony-Iron, 17.27 Tons TNT, Waktu: 1797-01-01 00:00:00
- Jepara (2008): 499.50 kg, Iron / Stony-Iron, 17.25 Tons TNT, Waktu: 2008-01-01 00:00:00
- Djati-Pengilon (1884): 166.00 kg, Ordinary Chondrite, 5.73 Tons TNT, Waktu: 1884-01-01 00:00:00


### 2.5 Penyeragaman Format Keluaran (Export CSV dengan UTF-8 Encoding)
Seluruh dataset yang telah melalui proses pembersihan, flattening, penyeragaman waktu, dan standardisasi skema diekspor seragam ke format **CSV (`.csv`)** dengan encoding **`UTF-8`** eksplisit.

In [9]:
# ==============================================================================
# 2.5 Export Seluruh Dataset ke Format CSV (UTF-8 Encoding)
# ==============================================================================

# Bersihkan file legacy di direktori output
for legacy_file in TRANSFORMED_DIR.glob("*.json"):
    legacy_file.unlink()

# 1. Meteorit Global Cleaned & Transformed
out_meteorites = TRANSFORMED_DIR / "meteorites_cleaned_transformed.csv"
df_clean_meteorites.to_csv(out_meteorites, index=False, encoding="utf-8")

# 2. Meteorit Indonesia Transformed
out_indonesia = TRANSFORMED_DIR / "indonesia_meteorites_transformed.csv"
df_indonesia.to_csv(out_indonesia, index=False, encoding="utf-8")

# 3. CNEOS Fireballs Transformed
out_fireballs = TRANSFORMED_DIR / "cneos_fireballs_transformed.csv"
df_clean_fireball.to_csv(out_fireballs, index=False, encoding="utf-8")

# 4. JPL Sentry Risk Objects Transformed
out_sentry = TRANSFORMED_DIR / "sentry_risks_transformed.csv"
df_clean_sentry.to_csv(out_sentry, index=False, encoding="utf-8")

# 5. SBDB CAD Close Approaches Transformed
out_cad = TRANSFORMED_DIR / "cad_approaches_transformed.csv"
df_clean_cad.to_csv(out_cad, index=False, encoding="utf-8")

# 6. NASA NeoWs Asteroids Flattened & Transformed
out_neows = TRANSFORMED_DIR / "neows_asteroids_transformed.csv"
df_neows.to_csv(out_neows, index=False, encoding="utf-8")

print("=== RINGKASAN DATASET TERSTANDARISASI (FORMAT SERAGAM: CSV, ENCODING: UTF-8) ===")
summary_rows = []
for f in sorted(TRANSFORMED_DIR.glob("*.csv")):
    df_check = pd.read_csv(f, encoding="utf-8")
    size_kb = f.stat().st_size / 1024
    has_time = "datetime_utc" in df_check.columns
    sample_time = df_check["datetime_utc"].iloc[0] if has_time and len(df_check) > 0 else "-"
    summary_rows.append({
        "File CSV": f.name,
        "Baris": f"{len(df_check):,}",
        "Kolom": len(df_check.columns),
        "Format Waktu Seragam": sample_time,
        "Encoding": "UTF-8",
        "Ukuran": f"{size_kb:.1f} KB"
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

=== RINGKASAN DATASET TERSTANDARISASI (FORMAT SERAGAM: CSV, ENCODING: UTF-8) ===


                            File CSV  Baris  Kolom Format Waktu Seragam Encoding    Ukuran
      cad_approaches_transformed.csv  1,097     13  2026-10-15 14:38:00    UTF-8  203.9 KB
     cneos_fireballs_transformed.csv    888      6  2026-10-04 03:14:45    UTF-8   47.2 KB
indonesia_meteorites_transformed.csv     19     21  1797-01-01 00:00:00    UTF-8    3.8 KB
  meteorites_cleaned_transformed.csv 31,911     20  1880-01-01 00:00:00    UTF-8 6457.0 KB
     neows_asteroids_transformed.csv     44     13  2026-10-29 17:05:00    UTF-8    6.5 KB
        sentry_risks_transformed.csv  2,251      9  2022-08-24 00:00:00    UTF-8  200.8 KB
